In [1]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# ============================
# CONFIG
# ============================
CSV_PATH = "../../results/vggnet16/all_k_vggnet16.csv"
OUT_PDF  = "Average_verification_time_paper.pdf"

# Base dataframe
df = pd.read_csv(CSV_PATH)

# numeric safety
df["k"] = pd.to_numeric(df["k"], errors="coerce")
df["all_time"] = pd.to_numeric(df["all_time"], errors="coerce")

# k values: smallest -> full image
pixel_vals = np.array([
    1568,    # 1/32
    3136,    # 1/16
    6272,    # 1/8
    12544,   # 1/4
    25088,   # 1/2
    50176,   # 1
])

# Keep only the k values of interest and non-null times
base = df[df["k"].isin(pixel_vals)].dropna(subset=["all_time"]).copy()

base = base[~base["result"].isin(["error"])].copy()

def mean_by_k(frame):
    return (
        frame.groupby("k")["all_time"]
             .mean()
             .reindex(pixel_vals)
             .values
    )

# 0) All rows
avg_time_all = mean_by_k(base)

# 1) Safe and unsafe:
# exclude timeout False and error
safe_unsafe = base[~base["result"].isin(["timeout False", "error"])].copy()
avg_time_safe_unsafe = mean_by_k(safe_unsafe)

# 2) Safe only:
# exclude timeout False, error, sat False, sat True
safe_only = base[
    ~base["result"].isin(["timeout False", "error", "sat False", "sat True"])
].copy()
avg_time_safe_only = mean_by_k(safe_only)

import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["font.family"] = "times"
mpl.rcParams["mathtext.fontset"] = "cm"

label_fs = 18
tick_fs = 18
legend_fs = 15

x_vals = np.array(pixel_vals)
x_labels = [
    r"$^{1}\!/_{32}$",
    r"$^{1}\!/_{16}$",
    r"$^{1}\!/_{8}$",
    r"$^{1}\!/_{4}$",
    r"$^{1}\!/_{2}$",
    r"$1$",
]

fig, ax = plt.subplots(figsize=(5.0, 4.0), dpi=300)

ax.set_xscale("log", base=2)
ax.set_xticks(x_vals)
ax.set_xticklabels(x_labels)
ax.set_xlim(x_vals[0] / 1.1, x_vals[-1] * 1.1)

ax.set_ylim(200, 1000)
ax.set_yticks([200, 400, 600, 800, 1000])

ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.6)
ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)

# New line 2: Safe only
ax.plot(
    x_vals,
    avg_time_safe_only,
    marker="^",
    linewidth=2,
    markersize=7,
    color="black",
    linestyle=":",
    zorder=3,
    markeredgewidth=1.5,
    markerfacecolor="white",
    label="Safe only",
)

# New line 1: Safe and unsafe
ax.plot(
    x_vals,
    avg_time_safe_unsafe,
    marker="x",
    linewidth=2,
    markersize=7,
    color="black",
    linestyle="--",
    zorder=3,
    markeredgewidth=2,
    markerfacecolor="white",
    label="Safe & unsafe",
)

# Existing line: all rows
ax.plot(
    x_vals,
    avg_time_all,
    marker="o",
    linewidth=2,
    markersize=8,
    color="black",
    markeredgewidth=2,
    markerfacecolor="white",
    markeredgecolor="black",
    zorder=3,
    label="All instances",
)

ax.set_xlabel("Fraction of pixels perturbed", fontsize=label_fs)
ax.set_ylabel("Mean verification time (s)", fontsize=label_fs)
ax.tick_params(axis="both", labelsize=tick_fs)
ax.legend(fontsize=legend_fs, frameon=False)

ax.legend(
    fontsize=14,      # text size
    frameon=False,
    handlelength=1.8, # line length in legend
    markerscale=0.9,  # marker size relative to plot marker
    labelspacing=0.3, # vertical space between entries
    borderpad=0.2,    # padding inside legend box
)

fig.tight_layout()
fig.savefig(OUT_PDF, format="pdf", bbox_inches="tight")
plt.show()


/var/folders/fr/3yg6c3f55w58dtlsnjcqjmth0000gq/T/ipykernel_20849/3800077789.py:152: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [2]:
COMBINED_OUT_PDF = "Average_verification_time_and_outcomes_paper.pdf"
L0_CSV_PATH = "../5.5.1_L0/imagenet_object_ratio_with_k_result_tag.csv"

merged = pd.read_csv(L0_CSV_PATH)

l0 = merged.copy()
l0["safe"] = (l0["result"] == "safe").astype(int)
l0["unsafe"] = (l0["result"] == "unsafe").astype(int)
l0["unknown"] = (l0["result"] == "unknown").astype(int)

whole_image = l0[l0["tag"] == "whole-image"].copy()
whole_image["budget_frac"] = whole_image["k"] / whole_image["total_pixel"]

outcome_summary = (
    whole_image.groupby("budget_frac", as_index=False)
    .agg(
        n=("result", "size"),
        safe=("safe", "sum"),
        unsafe=("unsafe", "sum"),
        unknown=("unknown", "sum"),
    )
    .sort_values("budget_frac")
)

outcome_summary["safe_rate"] = outcome_summary["safe"] / outcome_summary["n"]
outcome_summary["unsafe_rate"] = outcome_summary["unsafe"] / outcome_summary["n"]
outcome_summary["unknown_rate"] = outcome_summary["unknown"] / outcome_summary["n"]

def frac_label(x):
    if np.isclose(x, 1 / 32):
        return "1/32"
    if np.isclose(x, 1 / 16):
        return "1/16"
    if np.isclose(x, 1 / 8):
        return "1/8"
    if np.isclose(x, 1 / 4):
        return "1/4"
    if np.isclose(x, 1 / 2):
        return "1/2"
    if np.isclose(x, 1):
        return "1"
    return f"{x:.4f}"

order = ["1/32", "1/16", "1/8", "1/4", "1/2", "1"]
outcome_summary["label"] = outcome_summary["budget_frac"].apply(frac_label)
outcome_summary["label"] = pd.Categorical(outcome_summary["label"], categories=order, ordered=True)
outcome_summary = outcome_summary.sort_values("label")

outcome_label_map = {
    "1/32": r"$^{1}\!/_{32}$",
    "1/16": r"$^{1}\!/_{16}$",
    "1/8": r"$^{1}\!/_{8}$",
    "1/4": r"$^{1}\!/_{4}$",
    "1/2": r"$^{1}\!/_{2}$",
    "1": r"$1$",
}

outcome_x_vals = outcome_summary["budget_frac"].to_numpy(dtype=float)
outcome_x_labels = [outcome_label_map[str(label)] for label in outcome_summary["label"]]
max_rate = outcome_summary[["safe_rate", "unknown_rate", "unsafe_rate"]].to_numpy().max()

fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.0), dpi=300)
left_ax, right_ax = axes

left_ax.set_xscale("log", base=2)
left_ax.set_xticks(x_vals)
left_ax.set_xticklabels(x_labels)
left_ax.set_xlim(x_vals[0] / 1.1, x_vals[-1] * 1.1)
left_ax.set_ylim(200, 1000)
left_ax.set_yticks([200, 400, 600, 800, 1000])
left_ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.6)
left_ax.spines["top"].set_visible(True)
left_ax.spines["right"].set_visible(True)

left_ax.plot(
    x_vals,
    avg_time_safe_only,
    marker="^",
    linewidth=2,
    markersize=7,
    color="black",
    linestyle=":",
    zorder=3,
    markeredgewidth=1.5,
    markerfacecolor="white",
    label="Safe only",
)
left_ax.plot(
    x_vals,
    avg_time_safe_unsafe,
    marker="x",
    linewidth=2,
    markersize=7,
    color="black",
    linestyle="--",
    zorder=3,
    markeredgewidth=2,
    markerfacecolor="white",
    label="Safe & unsafe",
)
left_ax.plot(
    x_vals,
    avg_time_all,
    marker="o",
    linewidth=2,
    markersize=8,
    color="black",
    markeredgewidth=2,
    markerfacecolor="white",
    markeredgecolor="black",
    zorder=3,
    label="All instances",
)

left_ax.set_xlabel("Fraction of pixels perturbed", fontsize=label_fs)
left_ax.set_ylabel("Mean verification time (s)", fontsize=label_fs)
left_ax.tick_params(axis="both", labelsize=tick_fs)
left_ax.legend(
    fontsize=14,
    frameon=False,
    handlelength=1.8,
    markerscale=0.9,
    labelspacing=0.3,
    borderpad=0.2,
)

right_ax.set_xscale("log", base=2)
right_ax.set_xticks(outcome_x_vals)
right_ax.set_xticklabels(outcome_x_labels)
right_ax.set_xlim(outcome_x_vals[0] / 1.1, outcome_x_vals[-1] * 1.1)

right_ax.plot(
    outcome_x_vals,
    outcome_summary["safe_rate"],
    color="#111111",
    linestyle=":",
    linewidth=2,
    marker="^",
    markersize=7,
    markerfacecolor="white",
    markeredgewidth=1.5,
    label="Safe",
)
right_ax.plot(
    outcome_x_vals,
    outcome_summary["unsafe_rate"],
    color="#111111",
    linestyle="-",
    linewidth=2,
    marker="x",
    markersize=7,
    markeredgewidth=2,
    label="Unsafe",
)
right_ax.plot(
    outcome_x_vals,
    outcome_summary["unknown_rate"],
    color="#111111",
    linestyle="--",
    linewidth=2,
    marker="o",
    markersize=8,
    markerfacecolor="white",
    markeredgewidth=2,
    markeredgecolor="black",
    label="Unknown",
)

right_ax.set_xlabel("Fraction of pixels perturbed", fontsize=label_fs)
right_ax.set_ylabel("Fraction of instances", fontsize=label_fs)
right_ax.set_ylim(0, max_rate * 1.08)
right_ax.tick_params(axis="both", labelsize=tick_fs)
right_ax.set_axisbelow(True)
right_ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.6)
handles, labels = right_ax.get_legend_handles_labels()
legend_order = ["Safe", "Unsafe", "Unknown"]
ordered_handles = [handles[labels.index(label)] for label in legend_order]
right_ax.legend(
    ordered_handles,
    legend_order,
    loc="center left",
    bbox_to_anchor=(0.02, 0.5),
    fontsize=14,
    frameon=False,
    handlelength=1.8,
    markerscale=0.9,
    labelspacing=0.3,
    borderpad=0.2,
)
right_ax.spines["top"].set_visible(True)
right_ax.spines["right"].set_visible(True)

fig.tight_layout(w_pad=1.2)
fig.savefig(COMBINED_OUT_PDF, format="pdf", bbox_inches="tight")
plt.show()


/var/folders/fr/3yg6c3f55w58dtlsnjcqjmth0000gq/T/ipykernel_20849/591288045.py:195: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
